# MCP 工具接入设计

聚焦「工具池 + 外部 MCP 可插拔」的完整落地方案：
- 工具注册（声明式 schema）→ 工具池（registry + agent_tools 表）→ 三类传输协议
> → 调用时动态建连 → 启停热切换 / 重启自动重连 → 意图路由真实调用

### 背景

智能问答从「纯 RAG 检索回答」升级为「意图路由 + 工具调用」后，
工具池成为 Agent 的能力边界：内置工具（搜索/计算/时间）打底，
外部 MCP Server（高德地图、数据库、业务系统）可插拔扩展——
新增能力不改核心代码，「一切皆插件」。

### 一、整体架构（工具池模型）

```
┌─────────────────────────────────────────────────────────────────┐
│ ① 注册层（admin）                                               │
│   工具管理页：声明式 schema（名称 / 描述 / 参数 JSON Schema）   │
│   外部 MCP 注册：stdio 命令 / SSE URL / Streamable HTTP URL     │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│ ② 工具池（registry 内存池 + agent_tools 表持久化）              │
│   ToolSpec{name, description, parameters, handler, risk_level}  │
│   来源：builtin（内置）/ external（MCP）/ demo（本地演示）       │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│ ③ 调用层（意图路由选中工具 → execute_tool）                     │
│   内置工具：直接调 handler                                      │
│   外部 MCP：_SessionCtx 按 connect_config 动态建连 → call_tool  │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│ ④ 运营层（启停热切换 / 重启自动重连 / 审计）                    │
│   toggle 持久化 → 重启 sync_tools_from_db 恢复                   │
└─────────────────────────────────────────────────────────────────┘
```

**核心设计原则**：工具是"声明式插件"——注册即入库、入库即可路由、
调用时动态建连（不依赖注册时的 session 闭包，避免 Connection closed）。

### 二、完整链路

#### 路径 A：注册一个外部 MCP（一次性操作，admin）

| 步骤 | 发生什么 | 关键点 |
| --- | --- | --- |
| 1. 填写 | 工具管理页：名称 / 类型 / 端点或命令 / 前缀 | 三种协议选型 |
| 2. 校验 | 按 conn_type 校验（stdio 需 command，sse/http 需 url） | 后端 400 拦截 |
| 3. 拉取 | connect_mcp_* 建连 → list_tools 拉取工具列表 | 每次注册新建会话 |
| 4. 包装 | _wrap_external_tool 改名（前缀）+ 转 ToolSpec | 敏感词升级风险 |
| 5. 注入 | registry.register + upsert agent_tools 表 | 重启凭 connect_config 重连 |
| 6. 路由 | 智能问答意图识别选中该工具 → execute_tool 动态建连调用 | 零代码改动 |

#### 路径 B：启停热切换 / 卸载

| 动作 | 实现 |
| --- | --- |
| 停用 | registry.set_enabled(False) + DB enabled=False → 路由不再选中 |
| 启用 | set_enabled(True) + DB enabled=True → 立即恢复可用 |
| 卸载 | registry.unregister + DB 删除 → 工具池彻底移除 |
| 重启 | lifespan 里 sync_tools_from_db：恢复启停状态 + 自动重连启用中的外部 MCP |

### 三、传输协议对比

| 维度 | stdio | SSE | Streamable HTTP |
| --- | --- | --- | --- |
| 连接方式 | 本地子进程 stdin/stdout | 远程 EventSource 单向流 | HTTP 双向，会话复用（2026 主推） |
| 适用场景 | 本地 npx 启动的工具 | 老式远程 MCP 网关 | 云端工具（高德官方 MCP） |
| 会话管理 | 进程生命周期 | SSE 长连接 | HTTP 会话复用 |
| 本项目支持 | ✅ | ✅ | ✅（默认推荐） |

> 高德官方 2026 起推荐 Streamable HTTP，通用 SSE 已下线；
> 本项目三类协议走同一注册 API（conn_type 分支分发）。

### 四、关键代码

#### 1. 注册 API 与入参模型（`backend/app/api/tools.py`）

```python
class ExternalMcpRequest(BaseModel):
    name: str                                  # 连接名称（入库标识）
    conn_type: str = "stdio"                  # stdio | sse | streamable_http
    command: str = ""                         # stdio：外部 MCP 启动命令（如 npx @xxx/server）
    args: list[str] = []                      # stdio：命令参数
    url: str = ""                             # sse / streamable_http：端点 URL
    prefix: str = "ext_"                      # 工具名前缀，避免与内置冲突


@router.post("/external")
def register_external(req: ExternalMcpRequest, _user=Depends(require_admin)):
    # 1) 校验
    if req.conn_type not in ("stdio", "sse", "streamable_http"):
        raise HTTPException(status_code=400,
                            detail="conn_type 仅支持 stdio / sse / streamable_http")
    if req.conn_type == "stdio" and not req.command.strip():
        raise HTTPException(status_code=400, detail="stdio 模式必须提供 command")
    if req.conn_type in ("sse", "streamable_http") and not req.url.strip():
        raise HTTPException(status_code=400, detail=f"{req.conn_type} 模式必须提供 url")

    # 2) 按协议分发连接 + 拉取工具
    if req.conn_type == "stdio":
        registered = connect_mcp_stdio(req.command.strip(), req.args or [],
                                       prefix=req.prefix or "ext_")
        connect_config = {"type": "stdio", "command": req.command.strip(),
                          "args": req.args or []}
    elif req.conn_type == "sse":
        registered = connect_mcp_sse(req.url.strip(), prefix=req.prefix or "ext_")
        connect_config = {"type": "sse", "url": req.url.strip()}
    else:
        registered = connect_mcp_streamable_http(req.url.strip(),
                                                 prefix=req.prefix or "ext_")
        connect_config = {"type": "streamable_http", "url": req.url.strip()}

    if not registered:
        raise HTTPException(status_code=422,
                            detail="连接成功但未拉取到任何工具（检查 Server 是否实现了 tools/list）")

    # 3) 持久化（source=external + connect_config，重启重连凭据）
    db = SessionLocal()
    try:
        for tool_name in registered:
            spec = registry.get(tool_name)
            if spec is None:
                continue
            spec.source = "external"
            spec.connect_config = connect_config
            row = db.query(AgentToolConfig).filter(AgentToolConfig.name == tool_name).first()
            if row:
                row.enabled = True
                row.source = "external"
                row.connect_config = connect_config
            else:
                db.add(AgentToolConfig(name=tool_name, description=spec.description,
                                       risk_level=spec.risk_level, source="external",
                                       connect_config=connect_config, enabled=True))
        db.commit()
    except Exception as e:
        db.rollback()
        logger.error(f"持久化外部工具失败: {e}")
    finally:
        db.close()

    return {"ok": True, "registered": registered, "connect_config": connect_config}
```

#### 2. 三种协议连接 + 拉取工具（`backend/app/agent/tools/mcp.py`）以 Streamable HTTP 为例：

```python
def connect_mcp_streamable_http(url: str, prefix: str = "ext_") -> list[str]:
    '''连接 Streamable HTTP 型外部 MCP Server（如高德官方 MCP，2026 起推荐）'''
    from mcp import ClientSession
    from mcp.client.streamable_http import streamable_http_client

    connect_config = {"type": "streamable_http", "url": url}

    def _run() -> list[str]:
        async def _inner() -> list[str]:
            async with streamable_http_client(url) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await session.list_tools()
                    registered: list[str] = []
                    for t in tools.tools:
                        spec = _wrap_external_tool(t, prefix, connect_config)
                        try:
                            registry.register(spec)
                            registered.append(spec.name)
                        except ValueError as e:
                            logger.warning(f"外部工具注册跳过: {e}")  # 同名防冲突
                    return registered

        loop = asyncio.new_event_loop()
        try:
            return loop.run_until_complete(_inner())
        finally:
            loop.close()

    return _run()
```

#### 3. 调用时动态建连 —— 方案心脏（`_SessionCtx`）

```python
class _SessionCtx:
    '''按连接配置动态建立 MCP ClientSession（每次调用重建，避免闭包 session 失效）'''

    def __init__(self, connect_config: dict):
        self._connect_config = connect_config
        self._cm = None
        self.session = None

    async def __aenter__(self):
        from mcp import ClientSession
        ctype = self._connect_config.get("type", "streamable_http")
        if ctype == "stdio":
            from mcp import StdioServerParameters
            from mcp.client.stdio import stdio_client
            params = StdioServerParameters(
                command=self._connect_config.get("command", ""),
                args=self._connect_config.get("args") or [],
                env=None,
            )
            self._cm = stdio_client(params)
        elif ctype == "sse":
            from mcp.client.sse import sse_client
            self._cm = sse_client(self._connect_config.get("url", ""))
        else:
            from mcp.client.streamable_http import streamable_http_client
            self._cm = streamable_http_client(self._connect_config.get("url", ""))

        read, write = await self._cm.__aenter__()
        self.session = ClientSession(read, write)
        await self.session.__aenter__()
        await self.session.initialize()
        return self.session

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session is not None:
            await self.session.__aexit__(exc_type, exc_val, exc_tb)
        if self._cm is not None:
            await self._cm.__aexit__(exc_type, exc_val, exc_tb)
```

#### 4. 外部工具包装（改名 + schema + 风险升级）

```python
def _wrap_external_tool(tool, prefix: str, connect_config: dict) -> ToolSpec:
    '''把 MCP 工具声明包装成本地 ToolSpec（调用时按 connect_config 动态建连，转发 call_tool）'''
    name = f"{prefix}{tool.name}"
    parameters = dict(tool.input_schema or {"type": "object", "properties": {}, "required": []})
    # 外部工具默认按普通风险执行；描述含敏感词（删除/写/执行）的升级为 high
    risk = RISK_HIGH if any(w in tool.description.lower() for w in
                            ("delete", "remove", "write", "execute",
                             "send", "pay", "transfer")) else "normal"

    def handler(args: dict, ctx: ToolContext) -> Any:
        async def _call() -> Any:
            async with _SessionCtx(connect_config) as session:  # 每次动态建连
                result = await session.call_tool(tool.name, arguments=args)
            parts = [getattr(c, "text", None) for c in result.content]
            text = "
".join(p for p in parts if p)
            if getattr(result, "isError", False):
                raise RuntimeError(text or "外部 MCP 工具执行失败")
            return {"mcp_tool": tool.name, "result": text}

        loop = asyncio.new_event_loop()
        try:
            return loop.run_until_complete(_call())
        finally:
            loop.close()

    return ToolSpec(
        name=name,
        description=f"[MCP:{tool.name}] {tool.description}",
        parameters=parameters,
        handler=handler,
        risk_level=risk,
        tags=["mcp", "external"],
    )
```

#### 5. 启动自动重连（去重 + 前缀推导 + 失败停用）

```python
def sync_tools_from_db() -> None:
    '''启动时调用：把 DB 中保存的工具启停状态同步到内存池，并自动重连启用的外部 MCP'''
    db = SessionLocal()
    try:
        rows = db.query(AgentToolConfig).all()
    except Exception as e:
        logger.warning(f"读取工具配置失败（首次启动无表？）: {e}")
        return
    finally:
        db.close()

    seen_configs: set[tuple] = set()  # 同一连接配置只重连一次（DB 每工具一行，避免 N 次冗余连接）
    for row in rows:
        if row.source == "external" and row.enabled and row.connect_config:
            try:
                cfg = row.connect_config
                cfg_key = (cfg.get("type"), cfg.get("url"), cfg.get("command"))
                if cfg_key in seen_configs:
                    registry.set_enabled(row.name, True)  # 已重连，直接恢复启用
                    continue
                seen_configs.add(cfg_key)
                # prefix 从 DB 行名推导（amap_maps_weather -> amap_），
                # 保证重连注册的工具名与持久化行名一致，避免重复注册
                prefix = row.name.split("_", 1)[0] + "_"
                if cfg.get("type") == "stdio":
                    registered = connect_mcp_stdio(cfg.get("command", ""),
                                                   cfg.get("args") or [], prefix=prefix)
                elif cfg.get("type") == "sse":
                    registered = connect_mcp_sse(cfg.get("url", ""), prefix=prefix)
                else:
                    registered = connect_mcp_streamable_http(cfg.get("url", ""),
                                                             prefix=prefix)
                for n in registered:
                    sp = registry.get(n)
                    if sp:
                        sp.source = "external"
                        sp.connect_config = cfg
            except Exception as e:
                logger.error(f"自动重连外部 MCP 失败 {row.name}: {e}")
                registry.set_enabled(row.name, False)  # 重连失败默认停用，避免路由选中不可用工具
        else:
            registry.set_enabled(row.name, row.enabled)
    logger.info("工具池状态与 DB 同步完成")
```

#### 6. lifespan 调用（async 上下文必须丢线程池）

```python
# backend/app/main.py
from app.api.tools import sync_tools_from_db
# 同步函数内部用 asyncio.new_event_loop() 连接外部 MCP，
# 必须放到线程池执行，避免与 lifespan 的 running loop 冲突
await asyncio.to_thread(sync_tools_from_db)
```

#### 7. 前端注册弹窗（`frontend/src/pages/ToolsPage.jsx`）

```jsx
<Form.Item name="conn_type" label="连接类型">
  <Select options={[
    { value: 'streamable_http', label: 'Streamable HTTP（推荐，远程 URL 端点）' },
    { value: 'sse', label: 'SSE（远程 URL 端点）' },
    { value: 'stdio', label: 'stdio（本地命令启动，如 npx）' },
  ]} />
</Form.Item>

{/* connType === 'stdio' 显示「启动命令 + 参数」，否则显示「MCP Server 端点」 */}
{connType === 'stdio' ? (
  <>
    <Form.Item name="command" label="启动命令" rules={[{ required: true, message: '请输入启动命令' }]}>
      <Input placeholder="如：npx -y @modelcontextprotocol/server-github" />
    </Form.Item>
    <Form.Item name="args" label="命令参数（空格分隔，可选）">
      <Input placeholder="如：--token xxx" />
    </Form.Item>
  </>
) : (
  <Form.Item name="url" label="MCP Server 端点" rules={[{ required: true, message: '请输入端点 URL' }]}>
    <Input placeholder="如：https://mcp.amap.com/mcp?key=xxx" />
  </Form.Item>
)}
```

### 五、实战一：高德官方 MCP（Streamable HTTP）

#### 接入信息
- 端点：`https://mcp.amap.com/mcp?key=<AMAP_KEY>`（2026 起高德官方推荐 Streamable HTTP，通用 SSE 已下线）
- 注册请求：`POST /api/tools/external`，conn_type=`streamable_http`，url 带 key
- 前缀：`amap_`（避免与内置工具冲突）
- 结果：注入 15 个 `amap_maps_*` 工具（天气 / 地理编码 / 公交路线 / 驾车 / 骑行 / 步行 / POI 搜索等）

#### 真实路由调用（意图识别 → 工具池选中）
| 用户问题 | 选中工具 | 结果 |
| --- | --- | --- |
| 今天珠海天气怎么样？ | amap_maps_weather | 珠海市 晴 25°C |
| 从珠海站到广州南站怎么坐公交？ | amap_maps_direction_transit_integrated | 多条换乘方案 |
| 附近有什么咖啡店？ | amap_maps_poi_search | POI 列表（名称/地址/距离） |

> 高德 key 存于 agent_tools 表的 connect_config（url 内），
> 不在代码仓库硬编码；生产可挪环境变量 / 配置中心。

### 六、实战二：本地自建 Streamable HTTP MCP（界面注册全流程）

#### 1. 自建演示 MCP Server（mcp 2.x 的 MCPServer）

```python
from mcp.server.mcpserver import MCPServer

server = MCPServer("demo-quote")


@server.tool()
def get_demo_quote(topic: str = "生活") -> str:
    "返回一条中文语录（演示工具）"
    quotes = {"生活": "生活就像海洋……",
              "坚持": "锲而不舍，金石可镂。—— 荀子", ...}
    return quotes.get(topic, "暂无")


@server.tool()
def get_demo_currency_rate(currency: str = "USD") -> str:
    "返回模拟汇率（演示工具）"
    return f"{currency.upper()} 兑人民币参考汇率：7.12（模拟数据）"


if __name__ == "__main__":
    server.run(transport="streamable-http", host="127.0.0.1",
               port=8765, streamable_http_path="/mcp")
```

#### 2. 界面注册（工具管理 → 注册外部 MCP）

| 字段 | 填写 | 说明 |
| --- | --- | --- |
| 连接名称 | demo-quote | 入库标识 |
| 连接类型 | Streamable HTTP（推荐） | 表单自动切出「端点 URL」 |
| MCP Server 端点 | http://127.0.0.1:8765/mcp | 本地演示服务 |
| 工具名前缀 | dq_ | 避免与内置/高德冲突 |

点「连接并注册」→ 提示「连接成功，已注册 2 个工具」→ 工具池新增
`dq_get_demo_quote`、`dq_get_demo_currency_rate`（来源=外部 MCP）。

#### 3. 意图路由真实调用（零代码改动）
- 「帮我查一下美元兑人民币的汇率是多少」→ dq_get_demo_currency_rate → 7.12
- 「给我一条关于坚持的语录」→ dq_get_demo_quote → 荀子语录

**结论**：界面注册到可被路由调用全程零代码改动——「一切皆插件」。

### 七、验证结果（端到端实测）

| 验证项 | 结果 |
| --- | --- |
| 三类协议注册 | stdio / sse / streamable_http 走同一 API（conn_type 分支） |
| 高德接入 | 15 个 amap_maps_* 注入（9 内置 + 15 = 24） |
| 天气路由 | 「今天珠海天气怎么样？」→ amap_maps_weather |
| 公交路由 | 「珠海站到广州南站公交」→ direction_transit_integrated |
| 本地演示接入 | demo-quote → dq_get_demo_quote / dq_get_demo_currency_rate |
| 汇率路由 | 「美元兑人民币汇率」→ dq_get_demo_currency_rate → 7.12 |
| 重启自动重连 | 高德 + 演示均自动恢复启用（prefix 从 DB 行名推导） |
| 启停热切换 | toggle 立即生效 + DB 持久化 |

**一句话总结**：MCP 接入 = 声明式注册（三类协议统一 API）→ 工具池注入
（改名 + schema + 风险升级）→ 调用时动态建连（_SessionCtx）→
持久化 + 重启重连（sync_tools_from_db）——外部工具进池即可被意图路由使用，
全程零核心代码改动。